In [1]:
import os
from dotenv import load_dotenv

#load_dotenv(dotenv_path='.env')
load_dotenv()
GROQ_API_KEY = os.getenv("GROQ_API_KEY")

if not GROQ_API_KEY:
    raise RuntimeError(".env 파일에 GROQ_API_KEY 가 없습니다.")
print("GROQ_API_KEY 설정 확인 완료", GROQ_API_KEY[:2])


GROQ_API_KEY 설정 확인 완료 gs


#### Groq API로 호출 가능한 모델 목록 확인

- Groq는 OpenAI 호환 API이므로 `openai` SDK에 `base_url`만 Groq 주소로 지정하면 `client.models.list()`로 모델 목록을 조회할 수 있습니다.
- 목록 조회는 토큰을 사용하지 않습니다.
- 아래 `ChatOpenAI`에서 사용할 모델명이 목록에 있는지 확인하는 용도로도 사용할 수 있습니다.

In [2]:
# openai SDK 불러오기 (Groq도 OpenAI 호환 API라서 같은 SDK 사용)
from openai import OpenAI

GROQ_BASE_URL = "https://api.groq.com/openai/v1"  # Groq API 엔드포인트

# Groq 키와 엔드포인트로 클라이언트 생성 (GROQ_API_KEY는 load_dotenv()로 .env에서 읽어 둔 값)
client = OpenAI(api_key=GROQ_API_KEY, base_url=GROQ_BASE_URL)

# 호출 가능한 모델 목록 조회 (id 기준 오름차순 정렬)
models = sorted(client.models.list().data, key=lambda m: m.id)
print(f"조회된 모델 수: {len(models)}\n")
for m in models:
    print(f"{m.id:<45} {m.owned_by}")

조회된 모델 수: 11

allam-2-7b                                    SDAIA
canopylabs/orpheus-arabic-saudi               Canopy Labs
canopylabs/orpheus-v1-english                 Canopy Labs
meta-llama/llama-prompt-guard-2-22m           Meta
meta-llama/llama-prompt-guard-2-86m           Meta
openai/gpt-oss-120b                           OpenAI
openai/gpt-oss-20b                            OpenAI
openai/gpt-oss-safeguard-20b                  OpenAI
qwen/qwen3.8-27b                              Alibaba Cloud
whisper-large-v3                              OpenAI
whisper-large-v3-turbo                        OpenAI


In [3]:
from langchain_core.prompts import ChatPromptTemplate

# prompt
prompt = ChatPromptTemplate.from_messages(
    [ ("system", "당신은 개발자입니다.") , 
     ("user", "{input}") ]
)
print(type(prompt))
print(prompt)

<class 'langchain_core.prompts.chat.ChatPromptTemplate'>
input_variables=['input'] input_types={} partial_variables={} messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=[], input_types={}, partial_variables={}, template='당신은 개발자입니다.'), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['input'], input_types={}, partial_variables={}, template='{input}'), additional_kwargs={})]


In [4]:
prompt_text = prompt.format(input="파이썬은 무엇인가요? 자세하게 설명해주세요")
print(type(prompt_text))
print(prompt_text)

<class 'str'>
System: 당신은 개발자입니다.
Human: 파이썬은 무엇인가요? 자세하게 설명해주세요


In [5]:
from langchain_openai import ChatOpenAI 

#llm = ChatOpenAI(api_key=OPENAI_API_KEY, model_name="gpt-3.5-turbo-0125")

# Groq API를 사용하는 ChatOpenAI 인스턴스 생성
llm = ChatOpenAI(
    api_key=GROQ_API_KEY,
    base_url="https://api.groq.com/openai/v1",  # Groq API 엔드포인트
    model="openai/gpt-oss-120b",
    # model="qwen/qwen3.8-27b",
    temperature=0.7
)
print(type(llm))
print(llm.model_name)

<class 'langchain_openai.chat_models.base.ChatOpenAI'>
openai/gpt-oss-120b


In [6]:
try:
    response = llm.invoke(prompt_text)
    print(type(response)) # AIMessage
    print("응답:", response.content)
except Exception as e:
    print(f"오류 발생: {e}")

<class 'langchain_core.messages.ai.AIMessage'>
응답: ## 파이썬(Python)이란?

파이썬은 **고수준(high‑level), 인터프리터 기반, 동적 타이핑(dynamic typing)** 언어이며, **범용 프로그래밍 언어**로 설계되었습니다. 1991년 네덜란드의 프로그래머 **귀도 반 로썸(Guido van Rossum)**이 처음 발표했으며, 현재는 **오픈소스** 프로젝트로 전 세계 수천 명의 개발자가 유지·보수하고 있습니다. 파이썬은 **읽기 쉬운 문법**과 **풍부한 표준·서드파티 라이브러리** 덕분에 웹 개발, 데이터 과학, 인공지능, 자동화, 임베디드 시스템 등 거의 모든 소프트웨어 분야에서 널리 사용됩니다.

---

## 1. 파이썬의 주요 특징

| 특징 | 설명 | 장점 |
|------|------|------|
| **읽기 쉬운 문법** | 들여쓰기로 블록을 구분하고, 불필요한 기호(세미콜론, 중괄호 등)가 거의 없음 | 코드 가독성·유지보수성 향상 |
| **동적 타이핑** | 변수 선언 시 타입을 명시하지 않음 | 빠른 프로토타이핑·개발 속도 |
| **인터프리터** | 소스 코드를 바로 실행 (컴파일 단계가 없음) | REPL(대화형 쉘) 활용 가능, 즉시 피드백 |
| **멀티패러다임** | 절차적, 객체지향, 함수형 프로그래밍 지원 | 상황에 맞는 스타일 선택 가능 |
| **풍부한 표준 라이브러리** | `os`, `datetime`, `json`, `http`, `sqlite3` 등 200+ 모듈 제공 | 외부 의존성 최소화 |
| **플랫폼 독립성** | Windows, macOS, Linux, BSD, 심지어 임베디드(라즈베리 파이, 마이크로컨트롤러)에서도 실행 | 한 번 작성하면 거의 모든 환경에서 동작 |
| **확장성** | C/C++ 로 작성된 확장 모듈을 손쉽게 로드 가능 (`ctypes`, `cffi`, `cython` 등) | 성능이 중요한 부분은 네이티브 코드로 

## LCEL(LangChain Expression Language)
### Prompt + LLM 모델 + OutputParser

In [7]:
from langchain_core.output_parsers import StrOutputParser

output_parser = StrOutputParser()

In [8]:
# LCEL
chain = prompt | llm | output_parser
print(type(chain))
print(chain)

<class 'langchain_core.runnables.base.RunnableSequence'>
first=ChatPromptTemplate(input_variables=['input'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=[], input_types={}, partial_variables={}, template='당신은 개발자입니다.'), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['input'], input_types={}, partial_variables={}, template='{input}'), additional_kwargs={})]) middle=[ChatOpenAI(metadata={'lc_versions': {'langchain-core': '1.6.7', 'langchain': '1.4.3', 'langchain-openai': '1.6.7'}}, client=<openai.resources.chat.completions.completions.Completions object at 0x000002683ECA5D30>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x000002683ECB3620>, root_client=<openai.OpenAI object at 0x000002683EC9E2D0>, root_async_client=<openai.AsyncOpenAI object at 0x000002683ECB3170>, model_name='openai/gpt-oss-120b', temperature=0.7, model_kwargs={}, ope

In [9]:
response = chain.invoke({"input":"LangChain은 무엇인가요?"})

In [10]:
print(type(response)) #str
print(response)

<class 'langchain_core.messages.base.TextAccessor'>
**LangChain**은 **대규모 언어 모델(LLM)** 을 활용한 **응용 프로그램**을 보다 쉽고 효율적으로 구축할 수 있도록 도와주는 **프레임워크**입니다.  
주요 목표는 “LLM을 **단일 함수**가 아니라 **시스템**으로서 활용하도록** 하는 것입니다. 이를 위해 LangChain은 여러 **구성 요소**와 **도구**를 제공해, LLM을 **데이터베이스, API, 사용자 인터페이스, 워크플로우** 등과 자연스럽게 연결합니다.

---

## 1. 핵심 개념

| 구성 요소 | 역할 | 예시 |
|-----------|------|------|
| **Chains** | 여러 LLM 호출·프롬프트·처리를 순차·조건부로 연결 | “질문 → 검색 → 요약 → 답변” 파이프라인 |
| **Agents** | 목표를 달성하기 위해 **동적으로** 도구를 선택·호출 | “날씨를 알려줘” → 검색 도구 → LLM이 결과를 포맷 |
| **Memory** | 대화·세션의 상태를 기억하여 컨텍스트 유지 | 챗봇이 이전 대화를 기억하고 연속적인 답변 제공 |
| **Prompt Templates** | 재사용 가능한 프롬프트 구조 | “다음 문장을 한국어로 번역해 주세요: {text}” |
| **Document Loaders** | 다양한 소스(파일, 웹, DB 등)에서 텍스트를 로드 | PDF, CSV, 웹 페이지, Notion 등 |
| **Vector Stores** | 임베딩 벡터를 저장·검색해 **유사도 기반 검색** 구현 | Pinecone, FAISS, Chroma 등 |
| **Toolkits** | 외부 API·데이터베이스·시스템을 래핑한 도구 집합 | 검색 엔진, 계산기, 코드 실행기 등 |
| **Callbacks / Tracing** | 실행 흐름을 로깅·디버깅·모니터링 | LangSmith(추적 서비스)와 연동 |

---

## 2. 왜 L